# De l'ACP aux auto-encodeurs profonds avec PyTorch Lightning

Un auto-encodeur comprime son entrée en un code de petite taille (l'encodeur), puis la reconstruit depuis ce code (le décodeur) : sa cible est son entrée elle-même. Ce notebook en entraîne plusieurs sur MNIST (60 000 images d'entraînement et 10 000 de test, des chiffres manuscrits 28 × 28 en niveaux de gris), du plus simple au plus profond :

1. Un auto-encodeur linéaire
2. L'auto-encodeur linéaire face à l'ACP
3. Un auto-encodeur profond
4. L'espace latent
5. La détection d'anomalies
6. La taille du code
7. Pour aller plus loin : un auto-encodeur épars

Chaque partie réutilise les modèles des précédentes ; la dernière est facultative. Pas besoin de GPU : le notebook tourne en 20 à 30 minutes sur un processeur, dont la moitié pour la partie 6, qui entraîne six auto-encodeurs.

## Installation et imports

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

lightning.seed_everything(0)

## Chargement de MNIST

In [ ]:
donnees_entrainement = torchvision.datasets.MNIST("donnees", train=True, download=True)
donnees_test = torchvision.datasets.MNIST("donnees", train=False, download=True)
dim_entree = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde : ni centrés, ni réduits
X_train = donnees_entrainement.data.reshape(-1, dim_entree).float() / 255
y_train = donnees_entrainement.targets
X_test = donnees_test.data.reshape(-1, dim_entree).float() / 255
y_test = donnees_test.targets
print(X_train.shape, X_test.shape)


def affiche_images(*rangees: torch.Tensor, titres: list | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rangees[0])
  _, axes = plt.subplots(
    len(rangees), n, figsize=(n * 1.2, len(rangees) * 1.3), squeeze=False
  )
  for rangee, images in zip(axes, rangees):
    for ax, image in zip(rangee, images):
      ax.imshow(image.reshape(28, 28), cmap="gray_r")
      ax.axis("off")
  for ax, titre in zip(axes[0], [] if titres is None else titres):
    ax.set_title(titre)
  plt.show()


affiche_images(X_train[:10], titres=y_train[:10].tolist())

## Les outils de l'entraînement

Ils sont fournis, et servent à tout le notebook. `AutoEncodeur` est celui de la slide « Entraîner un auto-encodeur : la cible est l'entrée », avec une perte au choix et une perte de validation :

- `AutoEncodeur`, un `LightningModule` qui enveloppe un réseau et une perte : la cible de chaque image est l'image elle-même ; il initialise les poids du réseau comme Keras, ce qui évite que des unités ReLU du code meurent ;
- `entraine(modele, nb_epoques_max, nom)`, qui entraîne un `AutoEncodeur` par lots de 256 images, mesure la perte de validation sur les images de test et rend sa dernière valeur ;
- `predit(reseau, X)`, qui applique un réseau à toutes les lignes de `X`, lot par lot, sans calculer de gradient.

In [ ]:
class AutoEncodeur(lightning.LightningModule):
  """Enveloppe Lightning : l'entrée est aussi la cible à reconstruire."""

  def __init__(
    self, modele: nn.Module, perte: nn.Module, learning_rate: float = 1e-3
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele", "perte"])
    self.modele = modele
    self.perte = perte
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, dim_entree)
    # Initialisation de Glorot, celle de Keras : avec celle de PyTorch, des
    # unités ReLU d'un petit code meurent (nulles pour toutes les images)
    for couche in modele.modules():
      if isinstance(couche, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(couche.weight)
        nn.init.zeros_(couche.bias)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor], etape: str) -> torch.Tensor:
    (images,) = batch
    perte = self.perte(self(images), images)
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    return perte

  def training_step(self, batch: tuple[torch.Tensor], indice_lot: int) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


chargeur_entrainement = DataLoader(TensorDataset(X_train), batch_size=256, shuffle=True)
# La perte de validation se calcule sur les images de test
chargeur_val = DataLoader(TensorDataset(X_test), batch_size=256)


def entraine(
  modele: AutoEncodeur, nb_epoques_max: int, nom: str, verbeux: bool = True
) -> float:
  """Entraîne le modèle sur MNIST et rend sa dernière perte de validation."""
  trainer = lightning.Trainer(
    max_epochs=nb_epoques_max,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    # (Lightning refuse ce callback quand enable_progress_bar est False)
    callbacks=[TQDMProgressBar()] if verbeux else [],
    enable_checkpointing=False,
    enable_progress_bar=verbeux,
    enable_model_summary=False,
  )
  trainer.fit(modele, chargeur_entrainement, chargeur_val)
  return trainer.callback_metrics["perte_val"].item()


@torch.no_grad()
def predit(modele: nn.Module, X: torch.Tensor, taille_lot: int = 256) -> torch.Tensor:
  """Applique un réseau à chaque ligne de X, lot par lot."""
  device = next(modele.parameters()).device
  mode_entrainement = modele.training
  modele.eval()
  predictions = [modele(batch.to(device)).cpu() for batch in X.split(taille_lot)]
  # Rend au réseau son mode : fit ne le remet plus en mode entraînement
  modele.train(mode_entrainement)
  return torch.cat(predictions)

## 1. Un auto-encodeur linéaire

Sans fonction d'activation, l'encodeur et le décodeur ne sont que des applications linéaires (plus un biais) : c'est l'auto-encodeur de la slide « Auto-encodeur en PyTorch », sans couche cachée ni activation. Construisez :

- `encodeur` : un `nn.Sequential` d'une seule couche `nn.Linear`, de `dim_entree` entrées vers `dim_code = 20` sorties ;
- `decodeur` : un `nn.Sequential` d'une seule couche `nn.Linear`, de `dim_code` entrées vers `dim_entree` sorties ;
- `autoencodeur`, un `nn.Sequential` qui enchaîne les deux, sous les noms `encodeur` et `decodeur` ;
- `modele`, un `AutoEncodeur` qui enveloppe `autoencodeur` avec la perte `nn.MSELoss()`, dont vous affichez le résumé avec `ModelSummary`.

La cellule après la solution l'entraîne.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Linear(in_features, out_features)`, sans activation après : une couche linéaire
- `nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))` : un `nn.Sequential` construit sur un `OrderedDict` nomme ses sous-modules, qu'on retrouve ensuite en `autoencodeur.encodeur` et `autoencodeur.decodeur` ; `encodeur` et `decodeur` restent utilisables seuls
- `print(ModelSummary(modele, max_depth=2))` doit afficher 15,7 K paramètres pour l'encodeur (784 × 20 poids + 20 biais) et 16,5 K pour le décodeur (20 × 784 + 784)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dim_code = 20
encodeur = nn.Sequential(nn.Linear(..., ...))
decodeur = ...
autoencodeur = nn.Sequential(OrderedDict(encodeur=..., decodeur=...))
modele = AutoEncodeur(autoencodeur, perte=...)
print(ModelSummary(modele, max_depth=2))
```

### Solution

In [ ]:
dim_code = 20
encodeur = nn.Sequential(nn.Linear(dim_entree, dim_code))
decodeur = nn.Sequential(nn.Linear(dim_code, dim_entree))
autoencodeur = nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))
modele = AutoEncodeur(autoencodeur, perte=nn.MSELoss())
print(ModelSummary(modele, max_depth=2))

### Entraînement

La cible de chaque image est l'image elle-même, et la perte de validation se calcule sur les images de test.

In [ ]:
perte_val = entraine(modele, nb_epoques_max=10, nom="lineaire")
print(f"Perte de validation : {perte_val:.4f}")
decodeur_lineaire = decodeur  # gardé pour la partie 2
X_test_lineaire = predit(autoencodeur, X_test)
affiche_images(X_test[:10], X_test_lineaire[:10], titres=y_test[:10].tolist())

## 2. L'auto-encodeur linéaire face à l'ACP

La slide « Auto-encodeur linéaire et ACP » l'affirme : entraîné avec la MSE, un auto-encodeur linéaire à code de taille k retrouve le sous-espace des k premières composantes de l'ACP, mais pas ses axes. Vérifiez-le :

- ajustez une `PCA` à `dim_code` composantes sur `X_train`, reconstruisez `X_test` avec elle, et comparez l'erreur quadratique moyenne de sa reconstruction à celle de l'auto-encodeur (`X_test_lineaire`, calculé par la cellule précédente) ;
- les 20 directions du décodeur sont les colonnes du poids `weight` (784 × 20) de sa couche `nn.Linear` : mesurez, pour chacune, la part de sa norme qui tombe dans le sous-espace de l'ACP ;
- mesurez enfin le plus grand cosinus, en valeur absolue, entre deux directions du décodeur, puis entre deux composantes de l'ACP.

Les deux erreurs sont-elles proches ? Les directions du décodeur sont-elles dans le sous-espace de l'ACP ? Sont-elles orthogonales, comme les composantes ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`torch.Tensor.numpy`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- scikit-learn travaille sur des tableaux NumPy : `X_train.numpy()` en donne un, qui partage la mémoire du tenseur
- `acp = PCA(dim_code).fit(X_train.numpy())`, puis `acp.inverse_transform(acp.transform(X_test.numpy()))` : la reconstruction par l'ACP ; l'erreur : `np.mean((X_test.numpy() - X_test_acp) ** 2)`
- `nn.Linear` range ses poids en (sorties, entrées) : `decodeur_lineaire[0].weight` est de forme (784, 20), une direction par colonne ; `W = decodeur_lineaire[0].weight.detach().numpy().T` les met en lignes, de forme (20, 784), comme `acp.components_` (`.detach()` sépare le tenseur du calcul des gradients, ce que `.numpy()` exige)
- les lignes de `acp.components_` sont orthonormées : `W @ acp.components_.T` donne les coordonnées de chaque direction dans la base de l'ACP ; la norme de chaque ligne de ce produit (`np.linalg.norm(..., axis=1)`), divisée par celle de la ligne de `W`, est la part cherchée (1 : la direction est entièrement dans le sous-espace)
- les cosinus : normalisez les lignes, `V / np.linalg.norm(V, axis=1, keepdims=True)`, puis `V @ V.T` ; hors de la diagonale (`~np.eye(20, dtype=bool)`), 0 pour deux directions orthogonales

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
acp = PCA(...).fit(...)
X_test_acp = ...
erreur_acp = np.mean((X_test.numpy() - X_test_acp) ** 2)
erreur_lineaire = ...
print(f"Erreur : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}")

W = decodeur_lineaire[0].weight.detach().numpy().T
part = ...
print(f"Part dans le sous-espace de l'ACP : {part.min():.3f} à {part.max():.3f}")


def cosinus_max(V: np.ndarray) -> float:
  V = ...
  cosinus = ...
  return np.abs(cosinus[~np.eye(len(V), dtype=bool)]).max()


print(f"Plus grand cosinus : décodeur {cosinus_max(W):.2f}, ACP {cosinus_max(...):.2f}")
```

### Solution

In [ ]:
acp = PCA(dim_code).fit(X_train.numpy())
X_test_acp = acp.inverse_transform(acp.transform(X_test.numpy()))
erreur_acp = np.mean((X_test.numpy() - X_test_acp) ** 2)
erreur_lineaire = ((X_test - X_test_lineaire) ** 2).mean().item()
print(f"Erreur : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}")

W = decodeur_lineaire[0].weight.detach().numpy().T  # (20, 784)
part = np.linalg.norm(W @ acp.components_.T, axis=1) / np.linalg.norm(W, axis=1)
print(f"Part dans le sous-espace de l'ACP : de {part.min():.3f} à {part.max():.3f}")


def cosinus_max(V: np.ndarray) -> float:
  V = V / np.linalg.norm(V, axis=1, keepdims=True)
  cosinus = V @ V.T
  return np.abs(cosinus[~np.eye(len(V), dtype=bool)]).max()


print(
  f"Plus grand cosinus : décodeur {cosinus_max(W):.2f}, ACP {cosinus_max(acp.components_):.2f}"
)

Les deux erreurs sont presque égales (environ 0,024) : l'auto-encodeur linéaire fait aussi bien que l'ACP, mais pas mieux, puisque l'ACP donne la meilleure reconstruction linéaire possible à 20 dimensions. Ses 20 directions sont presque entièrement dans le sous-espace de l'ACP (de 93 % à presque 100 % de leur norme) : il a trouvé le même sous-espace, aux imprécisions de l'entraînement près. Mais elles ne sont pas orthogonales (le plus grand cosinus vaut environ 0,25, contre 0 pour l'ACP), ni triées par variance expliquée : rien dans la perte ne l'exige. L'ACP donne cette base en plus, et sans entraînement.

## 3. Un auto-encodeur profond

La slide « Auto-encodeur en PyTorch » ajoute une couche cachée non linéaire de part et d'autre du code, et finit le décodeur par une sigmoïde, entraînée avec l'entropie croisée binaire. Ce choix de sortie et de perte compte-t-il ? Pour le comparer à une sortie linéaire entraînée avec la MSE, écrivez `construit_autoencodeur(dim_code, activation_sortie)`, qui rend l'auto-encodeur profond, un `nn.Sequential` aux sous-modules nommés `encodeur` et `decodeur`, comme celui de la partie 1 :

- `encodeur` : une `nn.Linear` de `dim_entree` vers 128, puis une de 128 vers `dim_code`, chacune suivie d'une ReLU ;
- `decodeur` : une `nn.Linear` de `dim_code` vers 128 suivie d'une ReLU, puis une de 128 vers `dim_entree` suivie de `activation_sortie`, un module.

Le code a ici une ReLU, que la slide n'a pas : l'initialisation d'`AutoEncodeur` garde presque toutes ses unités en vie, et les parties 4 et 7 s'en servent. La cellule après la solution en entraîne deux, à code de `dim_code` nombres et pendant 10 epochs chacun : l'un à sortie linéaire avec la perte `nn.MSELoss()`, l'autre à sortie sigmoïde avec `nn.BCELoss()`. Elle compare leur erreur quadratique moyenne de reconstruction à celles de l'ACP et de l'auto-encodeur linéaire, et affiche la plus petite et la plus grande valeur de leurs sorties. Laquelle des deux sorties reconstruit le mieux ? Laquelle rend des pixels hors de [0, 1] ?

**Documentation utile**

- [`torch.nn.Identity`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Identity.html)

In [ ]:
def construit_autoencodeur(
  dim_code: int, activation_sortie: nn.Module
) -> nn.Sequential: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- l'`encodeur` et le `decodeur` de la slide, avec une `nn.ReLU()` après la couche du code et `activation_sortie` à la place de `nn.Sigmoid()`, puis `nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))`
- `activation_sortie` sera `nn.Identity()`, un module qui rend son entrée telle quelle (une sortie linéaire), ou `nn.Sigmoid()`, qui rend des valeurs dans ]0, 1[, comme les pixels
- `print(ModelSummary(AutoEncodeur(construit_autoencodeur(20, nn.Sigmoid()), perte=nn.BCELoss()), max_depth=2))` doit afficher 103 K paramètres pour l'encodeur (103 060) et 103 K pour le décodeur (103 824)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_autoencodeur(
  dim_code: int, activation_sortie: nn.Module
) -> nn.Sequential:
  encodeur = nn.Sequential(
    nn.Linear(dim_entree, 128), nn.ReLU(), nn.Linear(..., ...), nn.ReLU()
  )
  decodeur = nn.Sequential(..., ..., nn.Linear(128, dim_entree), ...)
  return nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))
```

### Solution

In [ ]:
def construit_autoencodeur(
  dim_code: int, activation_sortie: nn.Module
) -> nn.Sequential:
  encodeur = nn.Sequential(
    nn.Linear(dim_entree, 128),
    nn.ReLU(),
    nn.Linear(128, dim_code),
    nn.ReLU(),
  )
  decodeur = nn.Sequential(
    nn.Linear(dim_code, 128),
    nn.ReLU(),
    nn.Linear(128, dim_entree),
    activation_sortie,
  )
  return nn.Sequential(OrderedDict(encodeur=encodeur, decodeur=decodeur))

### Entraînement et comparaison

La suite du notebook garde l'auto-encodeur à sortie sigmoïde, `autoencodeur`, et son encodeur et son décodeur.

In [ ]:
sortie_lineaire = construit_autoencodeur(dim_code, nn.Identity())
entraine(
  AutoEncodeur(sortie_lineaire, perte=nn.MSELoss()),
  nb_epoques_max=10,
  nom="sortie_lineaire",
)
# Sortie dans ]0, 1[, comme les pixels, chacun traité comme une probabilité
autoencodeur = construit_autoencodeur(dim_code, nn.Sigmoid())
entraine(
  AutoEncodeur(autoencodeur, perte=nn.BCELoss()), nb_epoques_max=10, nom="profond"
)
encodeur, decodeur = autoencodeur.encodeur, autoencodeur.decodeur

X_test_sortie_lineaire = predit(sortie_lineaire, X_test)
X_test_pred = predit(autoencodeur, X_test)
print(
  f"Erreur quadratique moyenne : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}"
)
for nom, X_reconstruit in [
  ("AE profond, sortie linéaire et MSE", X_test_sortie_lineaire),
  ("AE profond, sigmoïde et BCE", X_test_pred),
]:
  erreur = ((X_test - X_reconstruit) ** 2).mean()
  print(
    f"{nom} : {erreur:.4f}, sorties de {X_reconstruit.min():.2f} à {X_reconstruit.max():.2f}"
  )
# En haut les originaux, puis l'AE linéaire, l'AE profond à sortie linéaire,
# et l'AE profond à sortie sigmoïde
affiche_images(
  X_test[:10],
  X_test_lineaire[:10],
  X_test_sortie_lineaire[:10],
  X_test_pred[:10],
  titres=y_test[:10].tolist(),
)

À 10 epochs chacun, la sigmoïde entraînée avec l'entropie croisée binaire reconstruit nettement mieux : une erreur d'environ 0,014, contre 0,023 pour la sortie linéaire, qui ne fait pas mieux que l'ACP ni que l'auto-encodeur linéaire. La sortie linéaire rend aussi des pixels hors de [0, 1] (d'environ -0,8 à 1,7), ce que la sigmoïde ne peut pas faire. D'où le fond gris de la deuxième et de la troisième rangée : `affiche_images` étale chaque image sur sa propre plage de valeurs, et un fond un peu négatif y paraît gris. Avec le même code de 20 nombres, l'auto-encodeur profond à sortie sigmoïde reconstruit près de deux fois mieux que l'ACP : ses couches non linéaires suivent des surfaces courbes, hors de portée d'un sous-espace.

## 4. L'espace latent

L'encodeur seul transforme chaque image de test en son code, un point d'un espace à `dim_code` dimensions : l'espace latent.

In [ ]:
codes = predit(encodeur, X_test)
print(codes.shape)

### Où vivent les codes ?

Le décodeur seul pourrait-il fabriquer de nouveaux chiffres à partir de codes tirés au hasard ? Regardez d'abord où vivent les vrais codes, puis essayez :

- pour chacune des 20 dimensions, affichez le minimum, la moyenne et le maximum des `codes`, et comptez les dimensions qui valent 0 pour toutes les images ;
- décodez 10 codes de bruit blanc, tirés uniformément dans [0, 1] sur chaque dimension, puis 10 codes tirés uniformément entre le minimum et le maximum de chaque dimension, et affichez les deux rangées d'images avec `affiche_images`.

Les codes ressemblent-ils à du bruit uniforme dans [0, 1] ? Les images décodées ressemblent-elles à des chiffres ? Pourquoi ?

**Documentation utile**

- [`torch.min`](https://docs.pytorch.org/docs/stable/generated/torch.min.html)
- [`torch.rand`](https://docs.pytorch.org/docs/stable/generated/torch.rand.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `codes.min(dim=0).values`, `codes.mean(dim=0)`, `codes.max(dim=0).values` : une valeur par dimension (`min` et `max` avec `dim` rendent aussi les indices, dans `.indices`) ; `.numpy().round(1)` pour les afficher
- une dimension qui vaut 0 pour toutes les images : `haut == 0` (la ReLU du code ne rend jamais moins de 0), puis `.sum()`
- `generateur = torch.Generator().manual_seed(0)`, puis `torch.rand(10, dim_code, generator=generateur)` : des valeurs uniformes dans [0, 1[ ; `bas + (haut - bas) * torch.rand(...)` les ramène entre `bas` et `haut`, dimension par dimension
- `predit(decodeur, bruit_blanc)` rend 10 images de 784 pixels, que `affiche_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bas, haut = ..., ...
print("Min :", bas.numpy().round(1))
print("Moyenne :", ...)
print("Max :", ...)
print("Dimensions mortes :", int(...))

generateur = torch.Generator().manual_seed(0)
bruit_blanc = torch.rand(...)
dans_plage = bas + (haut - bas) * ...
affiche_images(predit(decodeur, bruit_blanc), predit(decodeur, ...))
```

#### Solution

In [ ]:
bas, haut = codes.min(dim=0).values, codes.max(dim=0).values
print("Min :", bas.numpy().round(1))
print("Moyenne :", codes.mean(dim=0).numpy().round(1))
print("Max :", haut.numpy().round(1))
print("Dimensions mortes :", int((haut == 0).sum()))

generateur = torch.Generator().manual_seed(0)
bruit_blanc = torch.rand(10, dim_code, generator=generateur)
dans_plage = bas + (haut - bas) * torch.rand(10, dim_code, generator=generateur)
# En haut le bruit blanc, en bas les codes tirés dans leur plage
affiche_images(predit(decodeur, bruit_blanc), predit(decodeur, dans_plage))

Les codes ne ressemblent pas à du bruit uniforme dans [0, 1] : la ReLU les rend positifs ou nuls, leur plage change d'une dimension à l'autre et va jusqu'à plusieurs dizaines. Il arrive qu'une dimension soit morte : nulle pour toutes les images, elle ne porte aucune information. Le bruit blanc tombe donc dans un coin de l'espace latent, près de l'origine, que le décodeur n'a presque jamais vu : il rend des taches qui se ressemblent toutes. Tirés dans leur plage, les codes ne donnent pas mieux, des morceaux de traits qui ne forment pas de chiffre : les vrais codes n'occupent qu'une petite partie de ce pavé, car leurs dimensions sont liées entre elles. Un auto-encodeur compresse, mais ne génère pas : son décodeur ne sait décoder que les codes que produit l'encodeur. Pour générer, il faut imposer aux codes une loi connue dans laquelle on peut tirer : c'est ce que fait l'auto-encodeur variationnel (slide « Architecture d'un VAE »).

### Les centroïdes des chiffres

Les codes ne sont pas dispersés au hasard : ceux d'un même chiffre se regroupent. Le code moyen de chaque chiffre, son centroïde, décodé, en donne une image type.

In [ ]:
centroides = torch.stack(
  [codes[y_test == chiffre].mean(dim=0) for chiffre in range(10)]
)
affiche_images(predit(decodeur, centroides), titres=list(range(10)))

### Interpoler entre deux centroïdes

Les codes régulièrement espacés sur le segment qui relie deux centroïdes, une fois décodés, montrent comment le décodeur passe d'un chiffre à l'autre. Essayez d'autres paires.

In [ ]:
def parcours_latent(debut: int, fin: int, n: int = 12) -> None:
  # n codes régulièrement espacés, du centroïde de debut à celui de fin
  etapes = torch.linspace(0, 1, n)[:, None]
  parcours = centroides[debut] + etapes * (centroides[fin] - centroides[debut])
  affiche_images(predit(decodeur, parcours))


parcours_latent(7, 1)
parcours_latent(3, 8)

## 5. La détection d'anomalies

Un auto-encodeur reconstruit bien ce qui ressemble à ses exemples d'entraînement, et mal le reste : son erreur de reconstruction signale les anomalies (slide « Détection d'anomalies par erreur de reconstruction »). La cellule suivante prépare :

- `X_val` et `X_normal`, une image de test sur deux chacun : des chiffres normaux, que l'auto-encodeur n'a pas appris ;
- `X_anomalies`, 1 000 images de Fashion-MNIST, des vêtements : des anomalies pour un modèle de chiffres.

Puis, avec `autoencodeur` :

- calculez l'erreur quadratique moyenne de reconstruction de chaque image de `X_val`, de `X_normal` et de `X_anomalies` ;
- fixez le seuil au quantile à 99 % des erreurs de `X_val`, et mesurez la part des images de `X_normal`, puis de `X_anomalies`, qui le dépassent ;
- affichez au-dessus de leur reconstruction, avec `affiche_images`, les 10 images de `X_normal` les plus mal reconstruites, les 10 mieux reconstruites, et 10 vêtements.

Quelle part des chiffres normaux est signalée à tort ? Quelle part des vêtements est détectée ? Les pires chiffres sont-ils mal écrits, ou d'une classe précise ? Que fait le décodeur d'un vêtement ?

**Documentation utile**

- [`torch.quantile`](https://docs.pytorch.org/docs/stable/generated/torch.quantile.html)
- [`torch.argsort`](https://docs.pytorch.org/docs/stable/generated/torch.argsort.html)

In [ ]:
donnees_fashion = torchvision.datasets.FashionMNIST(
  "donnees", train=False, download=True
)
X_anomalies = donnees_fashion.data[:1000].reshape(-1, dim_entree).float() / 255
# Les images de test, en deux moitiés : l'une fixe le seuil, l'autre le teste
X_val, X_normal = X_test[::2], X_test[1::2]
print(X_val.shape, X_normal.shape, X_anomalies.shape)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- une petite fonction `erreurs_reconstruction(reseau, X)`, qui rend `((X - predit(reseau, X)) ** 2).mean(dim=1)` : une erreur par image (la moyenne sur ses 784 pixels), sert trois fois
- `torch.quantile(erreurs_val, 0.99)` : la valeur que dépassent 1 % des erreurs de validation. Pas celles des images d'entraînement : apprises, elles sont plus petites, d'où un seuil plus bas et plus de normaux signalés
- `(erreurs > seuil).float().mean()` : la part des images au-delà du seuil ; `f"{part:.1%}"` l'affiche en pourcentage
- `erreurs_normales.argsort(descending=True)` : les indices des images, de la plus grande erreur à la plus petite ; `affiche_images(X_normal[pires], predit(autoencodeur, X_normal[pires]))` : les originaux au-dessus de leur reconstruction

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def erreurs_reconstruction(reseau: nn.Module, X: torch.Tensor) -> torch.Tensor:
  return ...


seuil = torch.quantile(erreurs_reconstruction(autoencodeur, X_val), ...)
erreurs_normales = ...
erreurs_anomalies = ...
print(f"Seuil : {seuil:.4f}")
print(f"Chiffres normaux au-delà : {(erreurs_normales > seuil).float().mean():.1%}")
print(f"Vêtements au-delà : {...:.1%}")

ordre = erreurs_normales.argsort(descending=True)
pires, meilleurs = ordre[:10], ...
affiche_images(X_normal[pires], predit(autoencodeur, X_normal[pires]))
affiche_images(...)
affiche_images(X_anomalies[:10], ...)
```

### Solution

In [ ]:
def erreurs_reconstruction(reseau: nn.Module, X: torch.Tensor) -> torch.Tensor:
  # L'erreur quadratique moyenne de reconstruction de chaque image
  return ((X - predit(reseau, X)) ** 2).mean(dim=1)


# Le seuil vient de normaux de validation, que l'entraînement n'a pas vus
seuil = torch.quantile(erreurs_reconstruction(autoencodeur, X_val), 0.99)
erreurs_normales = erreurs_reconstruction(autoencodeur, X_normal)
erreurs_anomalies = erreurs_reconstruction(autoencodeur, X_anomalies)
print(f"Seuil : {seuil:.4f}")
print(f"Chiffres normaux au-delà : {(erreurs_normales > seuil).float().mean():.1%}")
print(f"Vêtements au-delà : {(erreurs_anomalies > seuil).float().mean():.1%}")

ordre = erreurs_normales.argsort(descending=True)
pires, meilleurs = ordre[:10], ordre[-10:]
print("Les chiffres les plus mal reconstruits")
affiche_images(X_normal[pires], predit(autoencodeur, X_normal[pires]))
print("Les mieux reconstruits")
affiche_images(X_normal[meilleurs], predit(autoencodeur, X_normal[meilleurs]))
print("Des vêtements")
affiche_images(X_anomalies[:10], predit(autoencodeur, X_anomalies[:10]))

Environ 1 % des chiffres normaux dépassent le seuil (0,9 % ici) : c'est ce que donne un quantile à 99 % fixé sur des images de la même sorte, que l'entraînement n'a pas vues non plus. Environ 90 % des vêtements le dépassent : le décodeur ne sait produire que des chiffres, et rend d'un vêtement une tache en forme de chiffre. Ceux qui passent sous le seuil sont surtout des chaussures fines, sandales et baskets : peu de pixels allumés, donc peu d'erreur. Les chiffres les plus mal reconstruits sont en général de forme rare (boucles inhabituelles, traits épais), que la reconstruction ramène vers un chiffre plus courant ; les mieux reconstruits, surtout des 1 : une forme simple, peu de pixels allumés. L'erreur dépend donc aussi de la classe : avec un seul seuil, un chiffre compliqué est plus souvent signalé qu'un 1, et un objet maigre passe plus facilement qu'un objet plein.

## 6. La taille du code

Un code plus grand reconstruit mieux, mais compresse moins (slide « Code sous-complet ou sur-complet »). Mesurez ce compromis, avec `construit_autoencodeur` de la partie 3 :

- entraînez un auto-encodeur profond à sortie sigmoïde pour chaque taille de code de 2, 4, 8, 16, 32 et 64, pendant 10 epochs chacun, avec la perte `nn.BCELoss()` et `entraine(..., verbeux=False)` ;
- gardez chaque réseau entraîné dans un dictionnaire `autoencodeurs`, et sa dernière perte de validation dans `pertes_val`, indexés par la taille du code ;
- tracez la perte de validation en fonction de la taille du code, avec un axe des abscisses en échelle logarithmique de base 2 ;
- affichez les 10 premières images de test, puis, en dessous, une rangée de leurs reconstructions par taille de code.

À partir de quelle taille les reconstructions sont-elles nettes ? Quelle taille de code choisiriez-vous ? Quel taux de compression (784 / taille du code) obtenez-vous alors ?

**Documentation utile**

- [`matplotlib.pyplot.xscale`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xscale.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `autoencodeurs[taille] = construit_autoencodeur(taille, nn.Sigmoid())`, puis `entraine(AutoEncodeur(autoencodeurs[taille], perte=nn.BCELoss()), nb_epoques_max=10, nom=f"taille_{taille}", verbeux=False)`, qui rend la dernière perte de validation ; `autoencodeurs[taille]`, le réseau enveloppé, est entraîné sur place
- la boucle prend quelques minutes : `print` la perte de chaque taille au fil de l'eau
- `plt.plot(list(pertes_val), list(pertes_val.values()), marker="o")`, puis `plt.xscale("log", base=2)`
- `affiche_images` prend autant de rangées qu'on lui en donne : `affiche_images(X_test[:10], *[predit(reseau, X_test[:10]) for reseau in autoencodeurs.values()])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
autoencodeurs, pertes_val = {}, {}
for taille in [2, 4, 8, 16, 32, 64]:
  autoencodeurs[taille] = ...
  pertes_val[taille] = entraine(...)
  print(f"Code de taille {taille} : {pertes_val[taille]:.4f}")

plt.plot(...)
plt.xscale(...)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

# En haut les originaux, puis une rangée par taille de code, de 2 à 64
affiche_images(X_test[:10], *[... for reseau in autoencodeurs.values()])
```

### Solution

In [ ]:
autoencodeurs, pertes_val = {}, {}
for taille in [2, 4, 8, 16, 32, 64]:
  autoencodeurs[taille] = construit_autoencodeur(taille, nn.Sigmoid())
  pertes_val[taille] = entraine(
    AutoEncodeur(autoencodeurs[taille], perte=nn.BCELoss()),
    nb_epoques_max=10,
    nom=f"taille_{taille}",
    verbeux=False,
  )
  print(
    f"Code de taille {taille} (compression {dim_entree / taille:.0f}x) : "
    f"perte de validation {pertes_val[taille]:.4f}"
  )

plt.plot(list(pertes_val), list(pertes_val.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

# En haut les originaux, puis une rangée par taille de code, de 2 à 64
affiche_images(
  X_test[:10],
  *[predit(reseau, X_test[:10]) for reseau in autoencodeurs.values()],
  titres=y_test[:10].tolist(),
)

La perte baisse vite tant que le code est très petit, puis de moins en moins : de 2 à 4 dimensions, elle perd environ 0,04, de 32 à 64, environ quatre fois moins. Les rangées le montrent : avec 2 nombres, les reconstructions sont floues et plusieurs chiffres en deviennent un autre (des 4 et des 5 rendus en 9) ; avec 8, les chiffres se reconnaissent ; à partir de 16 à 32, ils sont nets, et 64 n'y ajoute presque rien. Un code de 16 à 32 dimensions est un bon compromis, soit un taux de compression de 784 / 32 = 24,5 à 784 / 16 = 49 (en nombre de valeurs : le code et les pixels sont tous des flottants).

### Un code de 2 nombres face à l'ACP

Avec un code de 2 nombres, l'espace latent se dessine (slide « Espace latent d'un auto-encodeur »). La cellule suivante place chaque image de test en son code par `autoencodeurs[2]`, et, à côté, en ses deux premières composantes de l'ACP, l'autre réduction de dimension à 2, chaque point coloré selon son chiffre. Lequel des deux nuages sépare le mieux les chiffres ? Pourquoi ?

In [ ]:
codes_2d = predit(autoencodeurs[2].encodeur, X_test).numpy()
acp_2d = PCA(2).fit(X_train.numpy()).transform(X_test.numpy())

_, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, points_2d, titre in [
  (axes[0], codes_2d, "Auto-encodeur, code de 2"),
  (axes[1], acp_2d, "ACP, 2 composantes"),
]:
  for chiffre in range(10):
    points = points_2d[y_test.numpy() == chiffre]
    ax.scatter(points[:, 0], points[:, 1], s=2, label=chiffre)
  ax.set_title(titre)
  ax.legend(markerscale=5)
plt.show()

Le code de l'auto-encodeur sépare bien mieux les chiffres : les 0, les 1 et les 6 y forment chacun une branche à part, qui part de l'origine (la ReLU rend des codes positifs ou nuls) ; les 7 et les 9 se partagent le bas du nuage, et les 2, 3, 5 et 8 se chevauchent au milieu. Dans le plan de l'ACP, seuls les 0 et les 1 se détachent, aux deux bouts du premier axe ; le reste se mêle au centre. L'ACP garde les deux directions de plus grande variance des pixels, qui ne sont pas celles qui distinguent les chiffres ; l'encodeur, non linéaire, apprend deux nombres dont le décodeur doit pouvoir reconstruire chaque chiffre, et deux chiffres confondus dans le code ne le pourraient pas.

## 7. Pour aller plus loin : un auto-encodeur épars

Un auto-encodeur épars a un code large, mais une pénalité sur ses activations n'en laisse que peu actives pour chaque image : chaque unité du code finit par porter un motif qu'on peut regarder seul. Construisez-en un :

- `AutoEncodeurEpars`, une sous-classe d'`AutoEncodeur` dont `_step` ajoute à la perte de reconstruction une pénalité L1 sur les activations du code : `l1` fois la somme de leurs valeurs absolues, en moyenne sur les images du lot, avec `l1 = 1e-3` ;
- `autoencodeur_epars`, l'auto-encodeur profond de la partie 6 à code de 64 unités (`construit_autoencodeur(64, nn.Sigmoid())`), enveloppé dans un `AutoEncodeurEpars` avec la perte de la partie 3, et entraîné 10 epochs.

Comparez-le à l'auto-encodeur à code de 64 unités de la partie 6, `autoencodeurs[64]` : le nombre moyen d'unités actives (> 0) du code par image de test, le nombre d'unités jamais actives, et l'erreur quadratique moyenne de reconstruction. Que coûte la parcimonie ?

**Documentation utile**

- [`torch.Tensor.abs`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.abs.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- dans `_step`, `codes = self.modele.encodeur(images)`, puis `self.modele.decodeur(codes)` : la reconstruction, en passant par le code
- la pénalité : `self.l1 * codes.abs().sum(dim=1).mean()`, une somme sur les 64 unités de chaque image, puis une moyenne sur le lot ; c'est ce que fait Keras avec `activity_regularizer=keras.regularizers.L1(1e-3)` depuis sa version 3.14 (avant, il somme aussi sur le lot)
- `__init__` reçoit `l1` en plus, le garde dans `self.l1` et passe le reste à `super().__init__(modele, perte)`
- `codes = predit(reseau.encodeur, X_test)`, puis `(codes > 0).sum(dim=1).float().mean()` : le nombre moyen d'unités actives par image ; `(codes.max(dim=0).values == 0).sum()` : les unités jamais actives

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class AutoEncodeurEpars(AutoEncodeur):
  def __init__(self, modele: nn.Module, perte: nn.Module, l1: float) -> None:
    super().__init__(modele, perte)
    self.l1 = l1

  def _step(self, batch: tuple[torch.Tensor], etape: str) -> torch.Tensor:
    (images,) = batch
    codes = ...
    perte = self.perte(..., images) + ...
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    return perte


autoencodeur_epars = construit_autoencodeur(64, nn.Sigmoid())
entraine(AutoEncodeurEpars(...), nb_epoques_max=..., nom="epars")

for nom, reseau in [("Sans pénalité", autoencodeurs[64]), ("L1", autoencodeur_epars)]:
  codes = predit(reseau.encodeur, X_test)
  erreur = ((X_test - predit(reseau, X_test)) ** 2).mean()
  print(f"{nom} : ...")
```

### Solution

In [ ]:
class AutoEncodeurEpars(AutoEncodeur):
  """Un AutoEncodeur dont la perte ajoute une pénalité L1 sur les activations du code."""

  def __init__(self, modele: nn.Module, perte: nn.Module, l1: float) -> None:
    super().__init__(modele, perte)
    self.l1 = l1

  def _step(self, batch: tuple[torch.Tensor], etape: str) -> torch.Tensor:
    (images,) = batch
    codes = self.modele.encodeur(images)
    penalite = self.l1 * codes.abs().sum(dim=1).mean()
    perte = self.perte(self.modele.decodeur(codes), images) + penalite
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    return perte


autoencodeur_epars = construit_autoencodeur(64, nn.Sigmoid())
entraine(
  AutoEncodeurEpars(autoencodeur_epars, perte=nn.BCELoss(), l1=1e-3),
  nb_epoques_max=10,
  nom="epars",
  verbeux=False,
)

for nom, reseau in [("Sans pénalité", autoencodeurs[64]), ("L1", autoencodeur_epars)]:
  codes = predit(reseau.encodeur, X_test)
  erreur = ((X_test - predit(reseau, X_test)) ** 2).mean()
  print(
    f"{nom} : {(codes > 0).sum(dim=1).float().mean():.1f} unités actives par image sur 64, "
    f"{(codes.max(dim=0).values == 0).sum()} jamais actives, erreur {erreur:.4f}"
  )

Avec la pénalité, chaque image n'active plus qu'environ 25 unités du code au lieu d'environ 55, et près de la moitié des unités ne s'activent jamais : la pénalité éteint les unités les moins utiles. La parcimonie se paie en reconstruction, dont l'erreur double presque (de 0,007 à 0,012). Le coefficient règle ce compromis : essayez 1e-4 et 1e-2. Dans un code épars, chaque unité active porte un motif qu'on peut regarder seul : c'est ce qui sert à interpréter les représentations des grands modèles de langage.